# Prerequisites

In [1]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient
load_dotenv()

True

In [2]:
MONGODB_URI = os.getenv("MONGO_URI")
mongodb_client = MongoClient(MONGODB_URI)
mongodb_client.admin.command("ping")

{'ok': 1}

# Loading the dataset

In [3]:
import json

In [4]:
with open("dataset\\clean_data\\S08\\articles.json", 'r') as f:
    articles = json.load(f)

In [5]:
len(articles)

35

In [6]:
articles[0]

{'article': 'S08_set3_a4',
 'topic': 'abraham lincoln',
 'body': 'Abraham Lincoln\n\n\n\nAbraham Lincoln (February 12, 1809 â\x80\x93 April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion of slavery in the United States, "[I]n his short autobiography written for the 1860 presidential campaign, Lincoln would describe his protest in the Illinois legislature as one that \'briefly defined his position on the slavery question, and so far as it goes, it was then the same that it is now." This was in reference to the anti-expansion sentiments he had then expressed. Doris Kearns Goodwin, Team of Rivals: The Political Genius of Abraham Lincoln (2005) p. 91.  Holzer pg. 232.  Writing of the Cooper Union  speech, Holzer notes, "Cooper Union proved a unique confluence of political culture, rhetorical opportunity, technological innovation, and human genius, and it brought Abraham Lincoln to the

In [7]:
with open("dataset\\clean_data\\S08\\questions.json", 'r') as f:
    questions = json.load(f)

In [8]:
len(questions)

1471

In [9]:
questions[0]

{'topic': 'abraham_lincoln',
 'question': 'Was Abraham Lincoln the sixteenth President of the United States?',
 'answer': 'yes',
 'difficulty': {'questioner': 'easy', 'answerer': 'easy'},
 'article': 'S08_set3_a4'}

# Chunking and embedding articles

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import Dict, List
from voyageai.client import Client as VoClient
from tqdm import tqdm

In [11]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(model_name="gpt-4", chunk_size=200, chunk_overlap=0)

In [12]:
def get_chunks(doc: Dict, text_field: str) -> List[str]:
    text = doc[text_field]
    chunks = text_splitter.split_text(text)
    return chunks

In [15]:
vo = VoClient()

In [16]:
def get_embeddings(content: List[str], input_type: str):
    embds_obj = vo.contextualized_embed(inputs=[content], model='voyage-context-3', input_type=input_type)
    if input_type == "document":
        embeddings = [emb for r in embds_obj.results for emb in r.embeddings]
    if input_type == "query":
        embeddings = embds_obj.results[0].embeddings[0]
    return embeddings

In [18]:
embedded_articles = []
for article in tqdm(articles):
    chunks = get_chunks(article, 'body')
    chunk_embeddings = get_embeddings(chunks, 'document')
    for chunk, embedding in zip(chunks, chunk_embeddings):
        article_chunk = article.copy()
        article_chunk['body'] = chunk
        article_chunk['embedding'] = embedding
        embedded_articles.append(article_chunk)

100%|██████████| 35/35 [01:53<00:00,  3.23s/it]


In [19]:
len(embedded_articles)

1875

# Ingesting data into MongoDB database

In [21]:
DB_NAME = 'rag_discord'
COLLECTION_NAME = 'articles'
ATLAS_VECTOR_SEARCH_INDEX_NAME = 'vector_index'

In [22]:
collection = mongodb_client[DB_NAME][COLLECTION_NAME]

collection.delete_many({})

DeleteResult({'n': 0, 'electionId': ObjectId('7fffffff00000000000003d5'), 'opTime': {'ts': Timestamp(1770225455, 21), 't': 981}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1770225455, 21), 'signature': {'hash': b'\xe9-\xd6\xae\x82\x08\xed\xe9\xfd\x17\xfc\xcdz\xb6\x13\x17\xd0\x1d\x982', 'keyId': 7558919210034790401}}, 'operationTime': Timestamp(1770225455, 21)}, acknowledged=True)

In [23]:
collection.insert_many(embedded_articles)
print(f"Ingested {collection.count_documents({})} documents into the {COLLECTION_NAME} collection.")

Ingested 1875 documents into the articles collection.


# Creating a vector search index

In [24]:
from utils import create_index, check_index_ready

In [25]:
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine"
            }
        ]
    }
}

In [26]:
create_index(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME, model)

Creating the vector_index index


In [27]:
check_index_ready(collection, ATLAS_VECTOR_SEARCH_INDEX_NAME)

vector_index index status: PENDING
vector_index index status: PENDING
vector_index index status: PENDING
vector_index index status: READY
vector_index index definition: {'fields': [{'type': 'vector', 'path': 'embedding', 'numDimensions': 1024, 'similarity': 'cosine'}]}


# Performing vector search

In [28]:
def vector_search(user_query: str) -> List[Dict]:
    query_embedding = get_embeddings([user_query], "query")
    pipeline = [
        {
            "$vectorSearch": {
                "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 20,
                "limit": 5
            }
        },
        {
            "$project": {
                "_id": 0,
                "topic": 1,
                "body": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]
    
    results = collection.aggregate(pipeline)
    return results.to_list()

In [33]:
question = questions[5]
question

{'topic': 'abraham_lincoln',
 'question': 'Did his mother die of pneumonia?',
 'answer': 'No.',
 'difficulty': {'questioner': 'easy', 'answerer': 'easy'},
 'article': 'S08_set3_a4'}

In [34]:
vector_search(f"This question is about {question['topic']}. {question['question']}")

[{'topic': 'abraham lincoln',
  'body': 'Lincoln was just seven years old when, in 1816, the family was forced to make a new start in Perry County (now in Spencer County), Indiana. He later noted that this move was "partly on account of slavery," and partly because of difficulties with land deeds in Kentucky: Unlike land in the Northwest Territory, Kentucky never had a proper U.S. survey, and farmers often had difficulties proving title to their property. Lincoln was only nine when his mother, then thirty-four years old, died of milk sickness. Soon afterwards, his father remarried  to Sarah Bush Johnston. Sarah Lincoln raised young Lincoln like one of her own children. Years later she compared Lincoln to her own son, saying "Both were good boys, but I must say â\x80\x94 both now being dead that Abe was the best boy I ever saw or ever expect to see." Lincoln was affectionate toward his stepmother, whom he would call "Mother" for the rest of his life, but',
  'score': 0.7248650789260864}

# Implementing basic RAG

In [40]:
from openai import OpenAI

In [ ]:
def create_prompt(user_query: str) -> str:
    context = vector_search(user_query)
    context = "\n\n".join([article.get("body", "") for article in context])
    prompt = f"Context:\n{context}\n\nQuestion:{user_query}"
    return prompt

In [41]:
openAIClient = OpenAI()

def generate_answer(user_query: str) -> str:
    prompt = create_prompt(user_query)
    messages = [
        {"role": "developer", "content": "Answer the question based only on the provided context. If the context is empty, say I DON'T KNOW"},
        {"role": "user", "content": prompt}
        ]
    response = openAIClient.responses.create(
        model="gpt-5-nano",
        input=messages
    )
    return response.output_text

In [ ]:
question

{'topic': 'abraham_lincoln',
 'question': 'Did his mother die of pneumonia?',
 'answer': 'No.',
 'difficulty': {'questioner': 'easy', 'answerer': 'easy'},
 'article': 'S08_set3_a4'}

In [43]:
generate_answer(f"This question is about {question['topic']}. {question['question']}")

'No. His mother died of milk sickness, not pneumonia.'

# Adding chat history

In [44]:
from datetime import datetime

In [45]:
COLLECTION_HISTORY_NAME = "chat_history"

In [46]:
collection_history = mongodb_client[DB_NAME][COLLECTION_HISTORY_NAME]
collection_history.create_index("session_id")

'session_id_1'

In [47]:
def store_chat_message(session_id: int, role: str, content: str) -> None:
    message = {
        "session_id": session_id,
        "role": role,
        "content": content,
        "timestamp": datetime.now()
    }
    collection_history.insert_one(message)

In [48]:
def retrieve_session_history(session_id: int) -> List:
    cursor = collection_history.find({"session_id": session_id}).sort("timestamp", 1)
    if cursor:
        messages = [{"role": message['role'], "content": message['content']} for message in cursor]
    else:
        messages = []
    
    return messages

In [51]:
openAIClient = OpenAI()

In [73]:
def generate_answer(session_id: int, user_query: str):
    messages = []
    system_prompt = "Answer the questions based only on the provided context. If the context is empty, say I DON'T KNOW"
    messages.append({"role": "developer", "content": system_prompt})
    
    context = vector_search(user_query)
    context_str = "\n\n".join([chunk.get("body", "") for chunk in context])
    messages.append({"role": "user", "content": context_str})
    
    message_history = retrieve_session_history(session_id)
    messages.extend(message_history)
    
    user_message = {"role": "user", "content": user_query}
    messages.append(user_message)
    
    response = openAIClient.responses.create(
        model="gpt-5-nano",
        input=messages
    )
    
    store_chat_message(session_id, "user", user_query)
    store_chat_message(session_id, "assistant", response.output_text)
    
    return response.output_text

In [74]:
collection_history.delete_many({})

DeleteResult({'n': 0, 'electionId': ObjectId('7fffffff00000000000003d5'), 'opTime': {'ts': Timestamp(1770233935, 86), 't': 981}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1770233935, 86), 'signature': {'hash': b'\x82\x19\x90\x96\xe6Q\x16/\x8a\xe8\xa8\x9c\x07n\x9a\xa5\xfc\x8f\xef\x9c', 'keyId': 7558919210034790401}}, 'operationTime': Timestamp(1770233935, 86)}, acknowledged=True)

In [75]:
generate_answer(1, f"This question is about {question['topic']}. {question['question']}")

'No. The text states that his mother, Nancy Hanks Lincoln, died of milk sickness when he was nine.'

In [76]:
generate_answer(1, "What did I just ask you?")

'You asked: Did his mother die of pneumonia?'